<div class = "alert alert-info"> 
    <span style="font-family: 'Times New Roman'; font-size: 16px;">
    <h1 align = center> <b> A Listwise Approach for Dynamic Asset Selection </b> </h1>
    <h2 align = center> <font color = grey> ... </font> </h2>
    </span>
</div>

<span style="font-family: 'Times New Roman'; font-size: 16px;">

<h3> <b> Description </b> </h3>

Dans ce projet, nous allons  implémenter une approche avancée et innovante pour le ranking d'actions en fonction de leur attractivité, mesurée par leur potentiel de performance future. Nous nous appuyons sur une méthodologie de learning-to-rank (spécifiquement ListNet), qui est une technique de machine learning listwise permettant d'apprendre à classer des listes d'éléments de manière optimale, plutôt que de prédire des valeurs individuelles comme dans les modèles traditionnels. Cela le distingue des stratégies factorielles classiques (e.g., Fama-French), qui utilisent souvent des régressions linéaires ou des scores additifs pour sélectionner des actifs.

Cette approche pouurait être employé pour: l'optimisation de portefeuille (e.g., sélection top-k actions pour long-only), gestion dynamique (rebalancing mensuel), et remplacement de stratégies passives par une active apprenante.

</span>

<span style="font-family: 'Times New Roman'; font-size: 16px;">

<h3> <b> Données </b> </h3>

Nous allons travailler en utilisant comme univers le CAC40. Les données (prix historiques) seront téléchargées en utilisant l'API yahoo finance. A partir de ces données historiques, nous construirons de nouvelles variables (features) qui serviront de variables explicatives pour notre modèle. 


</br></br>



| **Feature**    | **Description détaillée**                                                                                                        | **Utilité pour le projet**                                                                                                  |
| -------------- | -------------------------------------------------------------------------------------------------------------------------------- | --------------------------------------------------------------------------------------------------------------------------- |
| **Momentum**   | Mesure la performance passée d’un actif sur une période donnée (ex : 12 mois – 1 mois) pour capturer les tendances persistantes. | Identifier les actions avec la plus forte probabilité de surperformance future (signal directionnel principal du modèle).   |
| **Volatilité** | Mesure de la dispersion des rendements (historique ou réalisée sur 20, 60, 120 jours, 1 an).                                           | Filtrer les actifs selon le régime de marché et ajuster la confiance des signaux (réduction du bruit).                      |
| **Beta 1 an**  | Sensibilité d’un actif aux mouvements du marché (corrélation + volatilité relative par rapport à un indice).                     | Contrôler l’exposition au risque systématique et réduire les biais liés aux mouvements de marché globaux.                   |
| **Volume**     | Nombre d’actions échangées sur une période (généralement moyenne 20 jours).                                                      | Mesure de liquidité permettant d’éviter les titres difficiles à trader (réduction du slippage et des coûts de transaction). |
| **ADX**        | Indicateur de tendance (Average Directional Index) mesurant la force d’un mouvement haussier ou baissier.                        | Séparer les tendances fortes du bruit de marché et améliorer la fiabilité du signal momentum.                               |
| **Regime**     | Probabilité que l'actif soit dans un régime de haute volatilité    |       Permet d'identifier les phases de hausse et de basse volatilité|



La variable cible que nous allons utiliser pour le ranking est la **rentabilité de l'actif du mois suivant**. Les actions seront donc classées selon leur **potentiel de performance du mois à venir**.

</span>

<span style="font-family: 'Times New Roman'; font-size: 16px;">

<h3> <b> Libraries </b> </h3>


</span>

In [ ]:
import pandas as pd
import numpy as np
import yfinance as yf
import statsmodels.api as sm
import pandas_ta as ta
from datetime import timedelta
from dateutil.relativedelta import relativedelta
import requests
from bs4 import BeautifulSoup


import warnings
warnings.filterwarnings('ignore')

<span style="font-family: 'Times New Roman'; font-size: 16px;">

<h3> <b> Data Acquisition & Preprocessing </b> </h3>

</span>

In [ ]:

# Liste des 40 tickers du CAC 40 (vérifier la composition si besoin)

tickers = [
    'AC.PA', 'AI.PA', 'AIR.PA', 'MT.AS', 'CS.PA', 'BNP.PA', 'EN.PA', 'BVI.PA',
    'CAP.PA', 'CA.PA', 'ACA.PA', 'BN.PA', 'DSY.PA', 'EDEN.PA', 'ENGI.PA', 'EL.PA',
    'ERF.PA', 'RMS.PA', 'KER.PA', 'OR.PA', 'LR.PA', 'MC.PA', 'ML.PA', 'ORA.PA',
    'RI.PA', 'PUB.PA', 'RNO.PA', 'SAF.PA', 'SGO.PA', 'SAN.PA', 'SU.PA', 'GLE.PA',
    'STLAP.PA', 'STMPA.PA', 'TEP.PA', 'HO.PA', 'TTE.PA', 'URW.PA', 'VIE.PA', 'DG.PA'
]
index_ticker = '^FCHI'

# Période des données historiques
start_date = '2005-01-01'
end_date = '2025-01-01'

# Téléchargement des données
print("Téléchargement des données historiques depuis Yahoo Finance...")
all_tickers = tickers + [index_ticker]
data = yf.download(all_tickers, start=start_date, end=end_date, auto_adjust=False, progress=False)

# MultiIndex: ['Adj Close','High','Low','Volume', ...]
prices = data['Adj Close']
highs = data['High']
lows = data['Low']
volumes = data['Volume']

# Rendements journaliers
returns = prices.pct_change().dropna(how='all')
if index_ticker not in returns.columns:
    raise KeyError(f"Le ticker de l'indice ({index_ticker}) n'a pas été téléchargé correctement.")
index_returns = returns[index_ticker]

# Resampling mensuel (month end)
monthly_prices = prices.resample('M').last()
monthly_returns = monthly_prices.pct_change().dropna(how='all')

# Fonction pour probabilité de régime (lookback 3 ans)
def get_regime_prob(returns_stock):
    try:
        ret_clean = returns_stock.dropna()
        if len(ret_clean) < 250:  # ~1 an de données journalières nécessaires pour estimation stable
            return np.nan
        model = sm.tsa.MarkovAutoregression(ret_clean, k_regimes=2, order=1, switching_variance=True)
        fit = model.fit(disp=False, maxiter=100)
        # smoothed_marginal_probabilities: DataFrame (t x n_regimes)
        probs = getattr(fit, 'smoothed_marginal_probabilities', None)
        if probs is None or probs.shape[1] < 2:
            return np.nan
        # Déterminer quel régime correspond à "haute volatilité" en regardant sigma2 si disponible
        try:
            p = fit.params
            # les noms peuvent différer selon la version; on essaye les clés usuelles
            var0 = p.get('sigma2.regime0', None) if isinstance(p, dict) else None
            var1 = p.get('sigma2.regime1', None) if isinstance(p, dict) else None
            # fallback: comparer variances conditionnelles moyennes par regime (approx)
            if var0 is None or var1 is None:
                # estimation simple: variance of residuals conditional (approx)
                # on prend la variance des résidus segmentée par prob>0.5
                resid = fit.resid
                mask = probs.iloc[:, 1] > probs.iloc[:, 0]
                var1_est = resid[mask].var() if mask.any() else 0
                var0_est = resid[~mask].var() if (~mask).any() else 0
                high_vol_reg = 1 if var1_est > var0_est else 0
            else:
                high_vol_reg = 1 if var1 > var0 else 0
        except Exception:
            high_vol_reg = 1  # fallback arbitraire
        return probs.iloc[-1, high_vol_reg]
    except Exception:
        return np.nan

# Construction de la base de données (panel mensuel pour 2010-2025)
print("Construction des features et variable cible pour 2010-2025...")
db_period_start = pd.to_datetime('2010-01-01')
db_period_end = pd.to_datetime('2025-01-01')

features_list = []
labels_list = []
dates_list = []

# Itérer sur les index de monthly_returns (début au 36ème mois pour ~3 ans de lookback)
for i in range(36, len(monthly_returns)):
    t = monthly_returns.index[i]

    # Filtre pour période 2010-2025
    if t < db_period_start or t >= db_period_end:
        continue

    # Tickers disponibles (non-NaN à la date t)
    mask_available = monthly_prices.loc[t].notna()
    available_tickers = [s for s in monthly_prices.columns[mask_available] if s != index_ticker and s in tickers]
    if not available_tickers:
        continue

    # Lookbacks
    start_1y = t - relativedelta(years=1)
    start_3y = t - relativedelta(years=3)
    start_12m = t - relativedelta(months=12)

    # Momentum: comparaison prix courant (t) vs prix ~12 mois avant (on prend nearest)
    prev_idx = monthly_prices.index.get_indexer([start_12m], method='nearest')[0]
    prev_12m = monthly_prices.iloc[prev_idx]
    mom = (monthly_prices.loc[t, available_tickers] / prev_12m[available_tickers]) - 1

    # Volatilité annualisée basée sur rendements journaliers sur 1 an (si dispo)
    try:
        vol = returns.loc[start_1y:t, available_tickers].std(ddof=1) * np.sqrt(252)
    except KeyError:
        # si certaines colonnes manquent dans returns, faire colonne par colonne
        vol = pd.Series({s: returns[s].loc[start_1y:t].std(ddof=1) * np.sqrt(252) for s in available_tickers})

    # Régime (probabilité d'être en régime haute vol)
    regime = pd.Series({s: get_regime_prob(returns[s].loc[start_3y:t]) for s in available_tickers})

    # Beta par rapport à l'indice (période 1 an)
    betas = {}
    idx_slice = index_returns.loc[start_1y:t].dropna()
    var_index = idx_slice.var() if len(idx_slice) > 1 else 0
    for s in available_tickers:
        r_s = returns[s].loc[start_1y:t].dropna()
        # aligner les deux séries sur l'intersection des dates
        common_idx = r_s.index.intersection(idx_slice.index)
        if len(common_idx) < 2 or var_index == 0:
            betas[s] = np.nan
        else:
            cov = r_s.loc[common_idx].cov(idx_slice.loc[common_idx])
            betas[s] = cov / var_index if var_index != 0 else np.nan
    betas = pd.Series(betas)

    # Avg Volume sur 1 an
    avg_vol = volumes.loc[start_1y:t, available_tickers].mean()

    # ADX: utiliser les 30 jours précédents (données journalières)
    adx = pd.Series(np.nan, index=available_tickers)
    end_date = t - timedelta(days=1)
    start_adx = end_date - timedelta(days=30)
    for s in available_tickers:
        # récupérer des séries synchrones high/low/close sur la fenêtre
        h = highs[s].loc[start_adx:end_date].dropna()
        l = lows[s].loc[start_adx:end_date].dropna()
        c = prices[s].loc[start_adx:end_date].dropna()
        # pour s'assurer que les trois ont au moins 14 observations et se recoupent
        common_idx = h.index.intersection(l.index).intersection(c.index)
        if len(common_idx) >= 14:
            df_ta = pd.DataFrame({'high': h.loc[common_idx], 'low': l.loc[common_idx], 'close': c.loc[common_idx]})
            try:
                adx_df = ta.adx(high=df_ta['high'], low=df_ta['low'], close=df_ta['close'], length=14)
                # Nom de colonne ADX_14
                if 'ADX_14' in adx_df.columns:
                    adx_val = adx_df['ADX_14'].iloc[-1]
                else:
                    # fallback : chercher colonne contenant 'ADX'
                    adx_col = [col for col in adx_df.columns if 'ADX' in col]
                    adx_val = adx_df[adx_col[-1]].iloc[-1] if adx_col else np.nan
                adx[s] = adx_val
            except Exception:
                adx[s] = np.nan

    # Features DF (index = tickers disponibles)
    features = pd.DataFrame({
        'Momentum': mom,
        'Volatilite': vol,
        'Regime': regime,
        'Beta': betas,
        'Avg_Volume': avg_vol,
        'ADX': adx
    })

    # Cible (rendements t+1 si disponible)
    if i + 1 < len(monthly_returns):
        target_row = monthly_returns.iloc[i + 1].reindex(features.index)
        # garder uniquement tickers avec target disponible
        valid_tickers = target_row[target_row.notna()].index.intersection(features.index)
        if len(valid_tickers) > 0:
            features = features.loc[valid_tickers].copy()
            target = target_row.loc[valid_tickers].copy()

            # Impute NaNs par la moyenne des colonnes (calcule sur les tickers restants)
            features = features.fillna(features.mean())

            # Stocker les DataFrames (garder les index tickers pour la construction finale)
            features_list.append(features)
            labels_list.append(target)
            dates_list.append(t)

# Construction du panel DF en préservant les tickers
panel_data = []
for features_df, target_series, dt in zip(features_list, labels_list, dates_list):
    temp_df = features_df.copy()
    temp_df['cible'] = target_series.values
    temp_df['date'] = dt
    temp_df['ticker'] = temp_df.index
    panel_data.append(temp_df.reset_index(drop=True))

if panel_data:
    full_db = pd.concat(panel_data, ignore_index=True)
else:
    full_db = pd.DataFrame(columns=['Momentum', 'Volatilite', 'Regime', 'Beta', 'Avg_Volume', 'ADX', 'cible', 'date', 'ticker'])

# Sauvegarde
full_db.to_csv('cac40_full_db_2010_2025.csv', index=False)

print("Base de données construite et sauvegardée dans 'cac40_full_db_2010_2025.csv'.")
print("Aperçu :")
print(full_db.head())


In [ ]:

# URL de la page Wikipedia CAC 40
url = "https://en.wikipedia.org/wiki/CAC_40"

var_agent = (
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
    "AppleWebKit/537.36 (KHTML, like Gecko) "
    "Chrome/124.0.0.0 Safari/537.36 "
    "Academic research; contact: youremail@gmail.com"
)

headers = {
    "User-Agent": var_agent,
    "Accept-Language": "en-US,en;q=0.9,fr;q=0.8",
    "Connection": "keep-alive"
}


# Téléchargement du HTML
response = requests.get(url, headers = headers)
soup = BeautifulSoup(response.text, "html.parser")

# Trouve le tableau de composition
# Ce tableau contient les en-têtes Company | Sector | GICS Sub-Industry | Ticker
table = None
for tbl in soup.find_all("table", {"class": "wikitable"}):
    # on vérifie si la première colonne mentionne "Company"
    header_row = tbl.find("tr")
    if header_row and "Company" in header_row.text and "Ticker" in header_row.text:
        table = tbl
        break

if table is None:
    raise ValueError("Tableau de composition introuvable sur la page Wikipedia.")

# Conversion en DataFrame
rows = []
headers = [th.get_text(strip=True) for th in table.find_all("th")]

for tr in table.find_all("tr")[1:]:
    cells = [td.get_text(strip=True) for td in tr.find_all("td")]
    if len(cells) == len(headers):
        rows.append(cells)

df_cac40 = pd.DataFrame(rows, columns=headers)

print(df_cac40)
df_cac40.to_csv("cac40_companies.csv", index = False)
